# U4-L4：接待来宾（视觉跟随与流程联调）

**课程目标**

1. 能将发现来宾、视觉跟随、到位确认与致意连接为一次接待流程，说明每一步的继续和退出条件。
2. 能复用L1的方向偏差、L2的实测框高阈值和L3的移动跟随，在来宾缓慢移动并停留后确认到位；目标不可用时阻断致意。
3. 能分别记录程序结束原因、致意任务状态及现场效果，结合跟随中的IMU记录和停稳后的里程计读数，解释机器人位移与朝向。


## 一、跟随结束后的接待任务

展厅来宾进入指定通道，机器人发现来宾后调整方向并缓慢接近。来宾沿标记路线移动，随后在接待点停留，机器人停稳后执行一次致意动作。

L3达到框高阈值后仍继续观察，来宾远离时还能重新跟进。本课需要确定什么时候结束跟随、交给致意环节：连续几次都达到停止占比且基本对准，才结束跟随；随后确认来宾位置与动作空间，再执行致意。连续满足条件用于减少一次检测波动带来的误触发，仍需现场确认来宾和机器人已经停稳。

接待流程为`发现单一来宾 → 视觉跟随 → 连续到位确认 → 再次检查 → 致意 → 结束`。方向和远近判断沿用L3，新增的是各步骤的交接与退出条件。IMU和里程计继续记录机器人运动，不用于决定来宾是否到位。


## 二、每一步怎样交接

| 步骤 | 继续条件 | 未满足时怎样处理 |
| --- | --- | --- |
| 等待来宾 | 有且只有一个可用目标框 | 无目标时有限等待；多目标、截断或处理异常则结束 |
| 跟随 | 每轮目标可用 | 远时跟进，偏差大时先转向，近时停止前进；目标不可用即结束 |
| 到位确认 | 连续3次达到停止占比且水平偏差不超过中心范围 | 任一次不满足，计数回到0；继续观察至本轮上限 |
| 致意前检查 | 来宾仍可用且到位，现场确认动作空间合适 | 不调用动作，记录具体原因 |
| 致意后收尾 | 查看句柄实际状态、核对物理动作 | 未成功不写成完成；保留未结束句柄，先处理旧任务 |

等待阶段不再另设“框高0.35就是3米”的触发比例。有一个可用来宾框即可进入跟随；用于到位的仍是L2实测停止占比，不由里程计推算来宾位置。

函数通过返回文字把结果交给主程序。调用者只有收到“到位”，才进入致意前检查。函数运行结束也可能是超时或目标无效，不能将所有返回都写成完成。


## 三、准备和静态联调

⚠️ **实验安全**：一位来宾按标记路线缓慢移动，面向相机、保持直立，随后短暂停留；不靠近机器人、不急转弯，不安排其他人员穿行。教师在侧方保护。转向、短步行走与致意都须事先核对空间，出现风险由人工立即中止。

📌 **前置**：L1～L3对应程序及阈值已核对；选择已验证的低幅度预定义动作；旧运动任务已结束。头部保持标定时角度，不同时启动语音、其他动作或跟随程序。

**STEP 01：首次连接，查看可用动作。** `greet_task`保存致意句柄，重复测试不重跑本格清掉旧句柄。


In [ ]:
import math
import time
from boosteros.robots.booster import BoosterRobot
from boosteros.brain import Detection

robot = BoosterRobot()
detector = Detection(model="person")
greet_task = None
for action in robot.list_actions():
    print(action.id, action.type, action.duration)
print("模式：", robot.get_mode())


### 3.1 复用已经核对的参数

**STEP 02：填写停止占比、跟随参数及致意动作。**

下格直接赋值。将`STOP_RATIO = 0.3`替换为本组L2实测值；其余跟随参数逐项采用L3已经核对的设置，不要因重新打开一课而照抄回示例值。尚未完成L3真机测试时，先完成L3测试。把目标、标定间距、头部角度和站姿填入§5.1记录表。若换用瓶子等目标，检测模型、`TARGET_CLASS`与标定值需一起调整；瓶子测试只用于核对程序分支。

跟随参数的含义与L3 STEP 02相同。本课新增：

| 参数 | 用途 |
| --- | --- |
| `GREET_ID` | 运行本格时输入STEP 01本机列表中、由教师确认的致意动作ID；不能输入列表序号。当前按整数ID读取，本机类型不同时先核对后调整 |
| `ARRIVAL_CHECKS` | 连续满足到位条件的次数，初值为3。它是观察次数，不是3秒，也不能证明来宾完全静止 |
| `ACTION_WAIT` | 致意任务的等待上限，单位秒；结合所选动作的实际时长由教师核对 |

本格最后初始化运动记录。重复接待时从STEP 10开始；需要改参数时可重跑本格，保留STEP 01中的`greet_task`原句柄。


In [ ]:
STOP_RATIO = 0.3         # 示例占位，必须替换为本组L2实测并核对的停止占比
TARGET_CLASS = "person"  # 更换目标时，检测模型、类别和阈值一起修改
FORWARD_SPEED = 0.10     # m/s，采用L3核对后的设置
KP_YAW = 0.001           # rad/s/像素，沿用L1、L3的偏差方向约定
MAX_YAW = 0.20           # rad/s，采用L3核对后的设置
CENTER_PX = 20           # 基本对准的像素范围
TURN_ONLY_PX = 80        # 偏差较大时只转向，不前进
PULSE_SECONDS = 0.20     # 秒，读IMU和里程计会增加指令持续时间
READ_LIMIT = 1.0         # 秒，取图与检测返回后检查，不能打断阻塞调用
RUN_SECONDS = 20.0       # 秒，一轮跟随的观察时限
print("停止占比：", STOP_RATIO)

GREET_ID = int(input("输入教师确认可用的致意动作ID："))
ARRIVAL_CHECKS = 3   # 连续满足次数，真机核对后可调整
ACTION_WAIT = 10.0  # 致意等待上限，按动作实际时长由教师核对
motion_rows = []
start_pose = None
end_pose = None


### 3.2 复用视觉观察与指令计算

`observe_target()`先取图并检查目标，再返回水平偏差、框高占比和说明；`choose_velocity()`据此返回前进速度、转向角速度和决定。方向符号、框高阈值、近时停止前进及偏差大时先转向，都沿用L3。

本格还保留了联调中的检测框数值检查：`math.isfinite()`判断数值是否为有限数，`all(...)`检查列出的各项是否都满足；宽、高还需大于0。这些检查会拒用异常框，不能识别人物身份或判断目标是否被遮挡。

定义这两个函数不会启动运动。先运行STEP 04核对静态分支，再连接后续流程。

**STEP 03：定义视觉观察与指令计算。**


In [ ]:
def observe_target():
    started = time.monotonic()
    image = robot.get_image(img_type="rgb")
    results = detector.detect(image.to_numpy())
    targets = []
    for result in results:
        if result.class_name == TARGET_CLASS:
            targets.append(result)
    if not targets:
        return None, None, "未检测到目标"
    if len(targets) != 1:
        return None, None, "检测到多个目标"

    box = targets[0].bbox
    values = [box.center_x, box.center_y, box.width, box.height]
    if not all(math.isfinite(value) for value in values):
        return None, None, "检测框包含无效数值"
    if box.width <= 0 or box.height <= 0:
        return None, None, "检测框大小无效"
    left = box.center_x - box.width / 2
    right = box.center_x + box.width / 2
    top = box.center_y - box.height / 2
    bottom = box.center_y + box.height / 2
    if left <= 2 or top <= 2 or right >= image.width - 2 or bottom >= image.height - 2:
        return None, None, "检测框碰到画面边缘，先检查目标是否完整"
    if time.monotonic() - started > READ_LIMIT:
        return None, None, "本次取图与检测过慢"
    error = box.center_x - image.width / 2
    ratio = box.height / image.height
    return error, ratio, "可用"


def choose_velocity(error, ratio):
    yaw_rate = 0.0
    if abs(error) > CENTER_PX:
        yaw_rate = max(-MAX_YAW, min(MAX_YAW, -KP_YAW * error))
    if ratio >= STOP_RATIO:
        return 0.0, yaw_rate, "已近，停止前进"
    if abs(error) > TURN_ONLY_PX:
        return 0.0, yaw_rate, "先原地对准"
    return FORWARD_SPEED, yaw_rate, "跟进"


**STEP 04：只读图，检查当前条件会怎样影响流程。** 让来宾分别在较远、偏左/右和接待间距处站定，记录拟发送的指令。未出现可用目标时不会进入跟随；已经到位仍需后续连续确认，不能只凭这一格就启动致意。


In [ ]:
error, ratio, note = observe_target()
if error is None:
    print("不能进入跟随：", note)
else:
    vx, yaw_rate, decision = choose_velocity(error, ratio)
    print(f"偏差={error:+.0f}px，占比={ratio:.3f}；{decision}")
    print(f"拟发vx={vx:.2f}m/s，vyaw={yaw_rate:+.3f}rad/s")
    print("本次是否满足到位条件：", ratio >= STOP_RATIO and abs(error) <= CENTER_PX)


### 3.3 场地与模式准备

标出起点、来宾移动路线和接待停留区，保持地面平整、目标完整可见。使用来宾并以1.2米标定时，最终停稳后用卷尺核对本组要求的间距。里程计会记录机器人经过的位置，不再输入卷尺起步间距来决定应该走多远。

**STEP 05：进入准备姿态。**


In [ ]:
robot.set_mode("prepare")
print("当前模式：", robot.get_mode())


> 人工支撑站起并确认站稳，核对头部角度、路线和动作范围后再继续。

**STEP 06：进入行走模式。**


In [ ]:
robot.set_mode("walk")
print("当前模式：", robot.get_mode())


## 四、连接接待流程

### 4.1 等待单一来宾

**STEP 07：定义有限等待。** `True`表示已出现可用来宾；程序结束或读数不可用时返回`False`。没有目标可以继续等，但多目标、截断或处理过慢需要先处理，不能随意选一个框继续。


In [ ]:
def wait_for_guest(seconds=10.0):
    started = time.monotonic()
    while time.monotonic() - started < seconds:
        error, ratio, note = observe_target()
        if error is not None:
            print("发现可用来宾，占比：", round(ratio, 3))
            return True
        if note != "未检测到目标":
            print("等待结束：", note)
            return False
        time.sleep(0.2)
    print("等待结束：规定时间内没有单一可用来宾")
    return False


### 4.2 跟随与连续到位确认

**STEP 08：定义跟随函数。** 函数的取图、计算指令和短步运动沿用L3。本课增加`arrived_count`，记录已经连续满足多少次到位条件。

一次到位条件是`ratio >= STOP_RATIO and abs(error) <= CENTER_PX`：框高达到停止占比，同时水平方向基本对准。两项都满足时发送零速度，计数加1；任一项不满足就把计数清零，按L3规则继续跟随。目标不可用则直接结束。到位确认期间只观察，不发送前进或转向指令。

例如，四轮检查依次为“满足、满足、不满足、满足”，计数依次为`1、2、0、1`，还不能进入致意。只有连续达到`ARRIVAL_CHECKS`次，函数才返回“到位”；达到观察时限仍未满足时，返回未完成原因。

IMU和里程计继续写入`motion_rows`，不参与到位判定。确认期间在零速度指令下读取，其他轮在短步末尾读取；发送零速度不等于已经物理停稳，停稳情况由现场核对。这里只定义函数，STEP 10才调用它开始运动。


In [ ]:
def follow_to_guest():
    global motion_rows, start_pose
    motion_rows = []
    robot.reset_odom()
    start_pose = list(robot.get_odom().pose_2d)
    arrived_count = 0
    started = time.monotonic()
    try:
        while time.monotonic() - started < RUN_SECONDS:
            error, ratio, note = observe_target()
            if error is None:
                return note
            vx, yaw_rate, decision = choose_velocity(error, ratio)
            at_goal = ratio >= STOP_RATIO and abs(error) <= CENTER_PX
            if at_goal:
                robot.set_velocity(0, 0, 0)
                arrived_count += 1
                decision = "到位确认"
            else:
                arrived_count = 0
                robot.set_velocity(vx, 0, yaw_rate)
            try:
                time.sleep(PULSE_SECONDS)
                wz = robot.get_imu().angular_velocity[2]
                x, y, yaw = robot.get_odom().pose_2d
            finally:
                robot.set_velocity(0, 0, 0)
            motion_rows.append([x, y, yaw, error, ratio, wz, decision])
            print(f"{decision}：偏差{error:+.0f}px，占比{ratio:.3f}，确认{arrived_count}/{ARRIVAL_CHECKS}")
            if arrived_count >= ARRIVAL_CHECKS:
                return "到位"
        return "跟随达到观察时限，未完成连续到位确认"
    finally:
        robot.set_velocity(0, 0, 0)


### 4.3 致意前再次确认

跟随返回“到位”后，先停止速度，再由教师确认停稳及动作空间。输入确认期间来宾仍可能移动，所以确认后重新取图；目标不可用、偏离中心或再次变远时跳过致意，不使用确认前的旧画面。

**STEP 09：定义致意检查与调用。** 只有实际状态`SUCCEEDED`才返回“致意任务成功”。等待超时会请求取消，并保留当前状态；是否已完整做出动作仍要现场记录。若本机动作采用等待后取消才释放占用，先记录实际行为和取消终态，交教师审定，不将`CANCELLED`自动改写成成功。


In [ ]:
def greet_guest():
    global greet_task
    if greet_task is not None and not greet_task.done():
        return "旧致意任务仍未结束"
    if input("已停稳且动作空间合适？输入“致意”：").strip() != "致意":
        return "未确认致意"
    if robot.get_mode() != "walk":
        return "当前模式不满足致意条件"
    error, ratio, note = observe_target()
    if error is None:
        return "致意前目标不可用：" + note
    if ratio < STOP_RATIO or abs(error) > CENTER_PX:
        return "来宾位置已改变，跳过致意"
    if GREET_ID not in [action.id for action in robot.list_actions()]:
        return "致意动作ID不在当前列表"
    greet_task = robot.do_action(GREET_ID)
    status = greet_task.wait(timeout=ACTION_WAIT)
    if not greet_task.done():
        greet_task.cancel()
        greet_task.wait(timeout=3.0)
        return "致意等待超时，取消后状态：" + str(greet_task.status)
    if status == "SUCCEEDED":
        return "致意任务成功"
    return "致意任务状态：" + str(status)


### 4.4 运行完整流程

**STEP 10：现场确认后执行一次接待。** 本格会调用前面的函数，实际转向、前进并可能致意。教师核对目标、阈值、路线、动作范围和模式后，将`APPROVED`改为`True`再运行；每次重复测试都要重新确认现场条件。

程序先等待单一可用来宾，再跟随；只有跟随函数返回“到位”，才进入致意前检查。到位后机器人保持零速度，教师确认停稳与空间并输入“致意”，程序再用新画面检查一次。这里的确认对应跟随转入动作的交接，仍需保留。

旧致意句柄未结束时不启动新一轮。其他连接的旧任务需事先人工核查，本程序的句柄检查不能覆盖它们。重复接待前先运行STEP 11确认旧任务结束，再恢复起点与现场条件；若已切回`prepare`，按STEP 05、06重新进入`walk`。

`finally`在正常退出或可处理的异常、中断时尝试发送零速度。若中断发生在动作等待期间，还需运行STEP 11检查动作句柄；零速度不能代替取消动作。断网或强制终止内核时按现场按钮流程处置。


In [ ]:
APPROVED = False
flow_reason = "本轮未运行"
motion_rows = []
start_pose = None
end_pose = None
if not APPROVED or robot.get_mode() != "walk":
    print("现场或模式条件未满足")
elif greet_task is not None and not greet_task.done():
    print("旧致意任务未结束，请先执行STEP 11")
else:
    flow_reason = "执行中断或异常，请核对报错与旧任务"
    try:
        robot.set_velocity(0, 0, 0)
        if wait_for_guest():
            flow_reason = follow_to_guest()
            print("跟随结果：", flow_reason)
            if flow_reason == "到位":
                flow_reason = greet_guest()
        else:
            flow_reason = "等待未获得单一可用来宾"
    finally:
        robot.set_velocity(0, 0, 0)
    print("本次接待结束：", flow_reason)


### 4.5 检查任务、记录终点，再收尾

**STEP 11：检查并结束仍活动的致意任务。** 保留原句柄；请求取消后仍未结束时，先由教师处理，不启动下一轮。发送停止指令、得到取消终态和机器人实际停稳是需要分别核对的事情。


In [ ]:
robot.set_velocity(0, 0, 0)
if greet_task is None:
    print("本轮没有致意句柄")
else:
    if not greet_task.done():
        greet_task.cancel()
        greet_task.wait(timeout=3.0)
    print("致意状态：", greet_task.status, "任务已结束：", greet_task.done())
print("请现场确认机器人已经停稳")


**STEP 12：停稳后读取最终位移与朝向。** `dx、dy`和方向角相对清零后里程计的参考轴；方向角表示起点到终点的方向，yaw表示机身朝向，两者可以不同。直线位移不等于沿途路程。这里的终点包括本轮致意可能产生的位置变化，不能全算作跟随运动。


In [ ]:
def report_motion(start_pose, end_pose, rows):
    dx = end_pose[0] - start_pose[0]
    dy = end_pose[1] - start_pose[1]
    displacement = math.hypot(dx, dy)
    print(f"参考坐标变化：dx={dx:+.3f} m，dy={dy:+.3f} m")
    print(f"起终点直线位移估计：{displacement:.3f} m（不是沿途路程）")
    if displacement >= 0.05:
        bearing = math.degrees(math.atan2(dy, dx))
        print(f"位移方向：相对里程计参考x轴{bearing:+.1f}°，正角朝参考y轴方向")
    else:
        print("位移较小，不解释方向角；0.05 m只是本次显示条件")
    print(f"起始/结束机身朝向：{start_pose[2]:+.3f}/{end_pose[2]:+.3f} rad")
    if rows:
        peak_wz = max(abs(row[5]) for row in rows)
        print(f"已记录{len(rows)}轮；采样中最大|z轴角速度|={peak_wz:.3f} rad/s")


if start_pose is None:
    print("本轮未进入跟随，没有起点记录")
else:
    end_pose = list(robot.get_odom().pose_2d)
    report_motion(start_pose, end_pose, motion_rows)
    print("本次结束原因：", flow_reason)


**STEP 13：回到准备姿态。** 先确认STEP 11任务已结束，再运行此格；未确认时先处理旧任务。随后按既定人工支撑与阻尼流程扶坐，禁止行走模式直接切阻尼。


In [ ]:
if greet_task is not None and not greet_task.done():
    print("致意任务尚未结束，请教师先处理")
else:
    robot.set_velocity(0, 0, 0)
    robot.set_mode("prepare")
    print("当前模式：", robot.get_mode())


## 五、接待记录与学习检查

### 5.1 按步骤核对

| 轮次/版本 | 来宾路线与标定条件 | 停止占比/速度/确认次数 | 等待结果 | 跟随结束原因 | 致意状态与现场表现 | 最终卷尺间距 |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | | | | | | |
| 2 | | | | | | |

正常接待之外，再安排一次来宾未出现或在开始前移出画面的测试；不能进入后续跟随或致意。运动中自然发生丢失、截断或多目标时保存输出，不为凑测试让人员突然穿进路线。

| 起始x/y/yaw | 终点x/y/yaw | 直线位移/方向 | 采样IMU角速度 | 当前任务是否结束 | 待改项 |
| --- | --- | --- | --- | --- | --- |
| | | | | | |

### 5.2 引用前三课的依据

- L1：记录用于本课的方向偏差符号、比例系数及转向上限；若方向不对，先处理再行走。
- L2：带入同一目标及拍摄条件的停止占比，引用静态采样与实测间距；里程计/卷尺实验用于理解位移估计误差。
- L3：引用移动目标的跟进、近处暂停、再次远离和小幅偏转记录；本课增加连续到位及致意交接，不重新证明身份跟踪或融合定位。

### 5.3 检查点与思考

1. 哪两项图像量决定跟随？为什么移动目标的接近不再由“起步间距减去1.2米”控制？
2. 连续到位计数什么时候增加、什么时候清零？连续3次满足能证明来宾完全静止吗？
3. 教师输入“致意”后为何还要再取图？来宾已经远离时函数会返回什么？
4. 里程计终点位移和IMU角速度能说明什么，为什么不能单独用它们证明已走到来宾跟前？
5. 任务返回`CANCELLED`，但现场看见动作完成，记录应怎样写？还需核对什么才能再次接待？

**本课完成依据**：一份真实接待记录、至少一项未满足条件时的阻断记录、停稳后的位移/朝向说明，以及旧任务退出记录。没有执行的步骤写明“未执行”。


## 六、扩展阅读：从课堂接待到应用

当前程序限定单一来宾、短路线和稳定拍摄条件，没有人物身份关联、障碍物识别或路线导航。来宾走动会改变身形和框大小，所以应从缓慢、面向相机的场景逐步验证，不能把一轮接待结果推广为任意人群中的自主服务。

相机控制方向和接近，IMU与里程计记录机身运动，信息在流程中有不同用途；当前没有将它们合成为统一的位置估计。后续U7再专门讨论多路信息的更新与多模态衔接，本课不提前引入订阅、线程或语音。

每次复测先固定目标、头部姿态、路线和参数，只改变一个待研究条件。若框高阈值附近反复切换，先检查检测波动与动作节奏；如果目标不完整或状态不可确认，应结束并重新检查，不能为完成接待而删掉条件。
